# Lesson 6 — Threshold Selection and Metric Choice

Read `README.md` (or `README.pl.md`) first — it has Meridian Outlet's framing for today. You saw real probabilities last lesson; now choose the threshold yourself.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import load_and_merge_orders

df = load_and_merge_orders()
df.shape

## Split, then split again

Same `train_df`/`test_df` split as Lesson 5. But this time, before you touch the model, carve a validation set out of `train_df` too — you'll compare thresholds on `val_df`, not on `test_df`. `test_df` stays untouched until the very last cell.

In [ ]:
from task import fit_classifier, split_for_validation, split_orders

train_df, test_df = split_orders(df)
fit_df, val_df = split_for_validation(train_df)
model = fit_classifier(fit_df)

## Lower the threshold, watch the predictions grow

At 0.5 almost nothing gets flagged. What happens at 0.3? At 0.2? All three predictions below are made on `val_df` — data the model saw during neither fitting nor (yet) final evaluation.

In [ ]:
from task import predict_at_threshold

predicted_05 = predict_at_threshold(model, val_df, 0.5)
predicted_03 = predict_at_threshold(model, val_df, 0.3)
predicted_02 = predict_at_threshold(model, val_df, 0.2)
predicted_05.sum(), predicted_03.sum(), predicted_02.sum()

## Precision, recall, and F1 at each threshold

Every extra flagged order is either a caught return (good) or a false alarm (a cost) — measured here on `val_df`.

In [ ]:
from task import classification_metrics

[
    classification_metrics(val_df["is_returned"], predicted)
    for predicted in [predicted_05, predicted_03, predicted_02]
]

## Your notes

Pick a threshold you'd actually recommend to Meridian Outlet — 0.5, 0.3, 0.2, or something else — and justify it in terms of the cost of a false positive (wrongly flagging a good order) versus a false negative (missing a real return). Base this only on what you saw on `val_df` above — you haven't looked at `test_df` yet.

_Write your answer here._

## The one-time reveal

Now that you've picked a threshold using validation data — not the test set — retrain on the *full* `train_df` (using every row you're allowed to train on, not just `fit_df`) and check your chosen threshold on `test_df`, exactly once. Compare this to what `val_df` predicted: is it close, or did it move a lot?

In [ ]:
CHOSEN_THRESHOLD = 0.2  # replace with your own choice from the val_df comparison above

final_model = fit_classifier(train_df)
final_predicted = predict_at_threshold(final_model, test_df, CHOSEN_THRESHOLD)
classification_metrics(test_df["is_returned"], final_predicted)